# Hybridmodell – Iteration 2: GraphSAGE + GRU + Fusion (Hurdle: binär & count)

Eigenes Modell für die hybride Link-Vorhersage. Zwei Ausgaben pro Paar (u→i) und
30-min-Fenster:
- **Binär** (Link ja/nein) → Vergleich gegen GraphMixer (AUC, AP)
- **Count** (Fahrtenzahl) → Vergleich gegen LSTM (MSE, MAE)

Architektur: Graph-Branch (**GraphSAGE** über statische Knoten-Features + Adjazenz)
+ Zeitreihen-Branch (**GRU** über die jüngste Verfügbarkeit je Station) → Fusion →
**zwei Köpfe** (Hurdle). Bewertung über das gemeinsame `shared_eval`-Protokoll.

Iteration 1 (GCN + 1D-CNN, nur binär) bleibt als dokumentierte Ablation erhalten.

## 1. Imports

In [ ]:
import os, sys, time
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Pfade
`ROOT` auf den Repo-/Projektordner zeigen lassen (Colab: vorher Drive mounten).

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# Pfade automatisch finden (funktioniert im Repo-Layout UND im Vault-Projektordner)
def _find(cands):
    for c in cands:
        if os.path.isdir(c): return os.path.abspath(c)
    return os.path.abspath(cands[0])
_GM = "temporal graph link prediciton (GraphMixer)"
PREP = _find([os.path.join("..","prepared Data"),       # Repo: hybrid_model/
              "prepared Data",                           # Repo-Wurzel
              os.path.join(_GM,"prepared"),              # Vault: Projektwurzel
              os.path.join("..",_GM,"prepared")])
EVAL = _find([os.path.join("..","evaluation"), "evaluation", os.path.join("..","..","evaluation")])
sys.path.insert(0, EVAL)
from shared_eval import SharedLinkEval, EvalConfig
for p, f in [(PREP,"node_static.npy"),(PREP,"node_avail.npy"),(PREP,"edge_index.npy"),
             (PREP,"edge_weight.npy"),(EVAL,"shared_eval.py")]:
    fp = os.path.join(p, f); print(("OK  " if os.path.isfile(fp) else "FEHLT ") + fp)

## 3. Konfiguration

In [ ]:
@dataclass
class Cfg:
    ts_lookback: int = 12      # GRU-Eingabe: letzte 12 Bins (6 h) Verfügbarkeit
    sage_hidden: int = 64
    sage_out: int = 64
    gru_hidden: int = 64
    fusion_hidden: int = 128
    dropout: float = 0.1
    lr: float = 1e-3
    epochs: int = 15
    batch_size: int = 1024
    lambda_count: float = 1.0  # Gewicht des Count-Loss
    seed: int = 42
cfg = Cfg()
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 4. Daten laden und normalisieren
Statische Features (z-Score), Verfügbarkeits-Zeitreihen (z-Score je Kanal, Stats
NUR aus dem Trainingszeitraum → kein Leakage).

In [ ]:
ev = SharedLinkEval()
bins_per_day = (24*60)//ev.cfg.bin_minutes
train_end_bin = ev.cfg.train_days*bins_per_day

node_static = np.load(os.path.join(PREP,"node_static.npy"))        # (N,3)
node_avail  = np.load(os.path.join(PREP,"node_avail.npy"))         # (N,T,4)
edge_index  = np.load(os.path.join(PREP,"edge_index.npy"))         # (2,E)
edge_weight = np.load(os.path.join(PREP,"edge_weight.npy"))        # (E,)
N, T, C = node_avail.shape
print("N,T,C =", N, T, C)

# z-Score statische Features
mu_s, sd_s = node_static.mean(0), node_static.std(0)+1e-6
static_x = ((node_static-mu_s)/sd_s).astype(np.float32)

# z-Score Verfügbarkeit je Kanal (nur Trainings-Bins)
tr = node_avail[:, :train_end_bin, :]
mu_a = tr.mean((0,1)); sd_a = tr.std((0,1))+1e-6
avail_n = ((node_avail-mu_a)/sd_a).astype(np.float32)

static_x_t = torch.tensor(static_x, device=device)

## 5. Normalisierte Adjazenz (für GraphSAGE)
Symmetrisierte, gewichtete Adjazenz (Gewicht = Trip-Frequenz im Training),
zeilenweise normalisiert → Mittelwert-Aggregation der Nachbarn.

In [ ]:
A = np.zeros((N,N), dtype=np.float32)
for (u,i),w in zip(edge_index.T, edge_weight):
    A[u,i]+=w; A[i,u]+=w           # symmetrisieren
deg = A.sum(1, keepdims=True)+1e-6
A_norm = torch.tensor(A/deg, device=device)   # (N,N) zeilen-normalisiert
print("A_norm:", tuple(A_norm.shape), "| mittlerer Grad:", float((A>0).sum(1).mean()))

## 6. Targets & Hilfsfunktionen
Kandidaten (Positives + Negatives) je Split aus `shared_eval`. Pro Kandidat
(u, i, bin) werden GRU-Fenster und Paar-Features gesammelt.

In [ ]:
# Paar-Frequenz (Training) fuer Paar-Feature
freq = {}
for (u,i),w in zip(edge_index.T, edge_weight):
    freq[(int(u),int(i))] = float(w)
WD_START = 3   # 2024-05-16 = Donnerstag (Mo=0)

# GRU-Eingaben (B, L, C); zero-gepolstert fuer fruehe Bins
def gather_windows(nodes, bins):
    B=len(nodes); L=cfg.ts_lookback
    out=np.zeros((B,L,C), dtype=np.float32)
    for k,(n,b) in enumerate(zip(nodes,bins)):
        lo=max(0,b-L); seg=avail_n[n, lo:b, :]
        if len(seg)>0: out[k, L-len(seg):, :]=seg
    return torch.tensor(out, device=device)

# Paar-Features: log1p(Frequenz), Distanz, zyklische Zeit (Stunde, Wochentag)
def pair_feats(u,i,b):
    B=len(u); f=np.zeros((B,6), dtype=np.float32)
    for k in range(B):
        fr=freq.get((int(u[k]),int(i[k])),0.0)
        dx=node_static[u[k],1]-node_static[i[k],1]; dy=node_static[u[k],2]-node_static[i[k],2]
        hour=((b[k]*ev.cfg.bin_minutes)//60)%24
        dow=(WD_START + (b[k]*ev.cfg.bin_minutes)//(60*24))%7
        f[k]=[np.log1p(fr), np.sqrt(dx*dx+dy*dy),
              np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
              np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7)]
    return torch.tensor(f, device=device)

cand_train = ev.build_candidates("train")
print("Train-Kandidaten:", len(cand_train), "| Positives:", int(cand_train["label"].sum()))

## 7. GraphSAGE-Branch
Zwei SAGE-Schichten: kombiniert eigenes Feature mit dem (normalisiert) aggregierten
Nachbar-Feature (Mean-Aggregation ueber `A_norm`).

In [ ]:
class SAGELayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin_self = nn.Linear(d_in, d_out)
        self.lin_neigh = nn.Linear(d_in, d_out)
    def forward(self, x, A_norm):
        neigh = A_norm @ x                       # Mean-Aggregation der Nachbarn
        return self.lin_self(x) + self.lin_neigh(neigh)

class GraphSAGE(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout):
        super().__init__()
        self.l1=SAGELayer(d_in,d_hidden); self.l2=SAGELayer(d_hidden,d_out)
        self.act=nn.ReLU(); self.do=nn.Dropout(dropout)
    def forward(self, x, A_norm):
        h=self.do(self.act(self.l1(x,A_norm)))
        return self.act(self.l2(h,A_norm))       # (N, d_out) Knoten-Embeddings

## 8. Hybridmodell (GRU + Fusion + zwei Köpfe)
GraphSAGE liefert pro Knoten ein Struktur-Embedding; der GRU verdichtet die
jüngste Verfügbarkeit. Für ein Paar werden beide Embeddings von u und i plus
Paar-Features fusioniert und auf zwei Köpfe geführt.

In [ ]:
class HybridHurdle(nn.Module):
    def __init__(self, cfg, n_static, n_channels, n_pair):
        super().__init__()
        self.sage = GraphSAGE(n_static, cfg.sage_hidden, cfg.sage_out, cfg.dropout)
        self.gru  = nn.GRU(n_channels, cfg.gru_hidden, batch_first=True)
        node_dim  = cfg.sage_out + cfg.gru_hidden
        fuse_in   = 2*node_dim + n_pair
        self.fusion = nn.Sequential(
            nn.Linear(fuse_in, cfg.fusion_hidden), nn.ReLU(), nn.Dropout(cfg.dropout))
        self.head_bin   = nn.Linear(cfg.fusion_hidden, 1)        # Logit
        self.head_count = nn.Linear(cfg.fusion_hidden, 1)        # -> Softplus
        self.softplus = nn.Softplus()
    def node_repr(self, sage_emb, idx, win):
        g,_ = self.gru(win)                       # (B,L,H)
        return torch.cat([sage_emb[idx], g[:,-1,:]], dim=-1)
    def forward(self, sage_emb, u, i, win_u, win_i, pf):
        hu=self.node_repr(sage_emb,u,win_u); hi=self.node_repr(sage_emb,i,win_i)
        z=self.fusion(torch.cat([hu,hi,pf], dim=-1))
        logit = self.head_bin(z).squeeze(-1)
        count = self.softplus(self.head_count(z)).squeeze(-1)
        return logit, count

model = HybridHurdle(cfg, n_static=3, n_channels=C, n_pair=6).to(device)
print("Parameter:", sum(p.numel() for p in model.parameters()))

## 9. Training
Gesamt-Loss = BCE (binär) + λ·MSE (count). GraphSAGE-Embeddings werden je Schritt
neu berechnet (kleiner Graph).

In [ ]:
opt=torch.optim.Adam(model.parameters(), lr=cfg.lr)
bce=nn.BCEWithLogitsLoss(); mse=nn.MSELoss()
u_all=cand_train["u"].to_numpy(); i_all=cand_train["i"].to_numpy()
b_all=cand_train["bin_idx"].to_numpy()
y_all=cand_train["label"].to_numpy().astype(np.float32)
c_all=cand_train["count"].to_numpy().astype(np.float32)
n=len(cand_train); rng=np.random.default_rng(cfg.seed)

for ep in range(1, cfg.epochs+1):
    model.train(); perm=rng.permutation(n); tot=0.0; nb=0
    for s in range(0, n, cfg.batch_size):
        bi=perm[s:s+cfg.batch_size]
        u,i,b=u_all[bi],i_all[bi],b_all[bi]
        wu=gather_windows(u,b); wi=gather_windows(i,b); pf=pair_feats(u,i,b)
        y=torch.tensor(y_all[bi],device=device); c=torch.tensor(c_all[bi],device=device)
        sage_emb=model.sage(static_x_t, A_norm)
        logit,count=model(sage_emb,u,i,wu,wi,pf)
        loss=bce(logit,y)+cfg.lambda_count*mse(count,c)
        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item(); nb+=1
    print(f"Epoche {ep:2d}/{cfg.epochs} | Loss {tot/max(1,nb):.4f}")

## 10. Vorhersage-Export & Bewertung
Erzeugt `score` (binär) und `pred_count` für die `shared_eval`-Kandidaten und
bewertet beide Köpfe – direkt vergleichbar mit GraphMixer (binär) und LSTM (count).

In [ ]:
@torch.no_grad()
def predict(split):
    model.eval()
    cand=ev.build_candidates(split)
    u=cand["u"].to_numpy(); i=cand["i"].to_numpy(); b=cand["bin_idx"].to_numpy()
    sage_emb=model.sage(static_x_t, A_norm)
    scores=np.zeros(len(cand),dtype=np.float32); counts=np.zeros(len(cand),dtype=np.float32)
    for s in range(0,len(cand),4096):
        sl=slice(s,s+4096)
        wu=gather_windows(u[sl],b[sl]); wi=gather_windows(i[sl],b[sl])
        pf=pair_feats(u[sl],i[sl],b[sl])
        logit,count=model(sage_emb,u[sl],i[sl],wu,wi,pf)
        scores[sl]=torch.sigmoid(logit).cpu().numpy(); counts[sl]=count.cpu().numpy()
    out=cand[["u","i","bin_idx"]].copy(); out["score"]=scores; out["pred_count"]=counts
    return out

out_dir=os.path.join(".","predictions"); os.makedirs(out_dir, exist_ok=True)
for split in ["val","test"]:
    pred=predict(split)
    pred.to_csv(os.path.join(out_dir,f"hybrid_pred_{split}.csv"), index=False)
    rb=ev.score_binary(pred, split=split); rc=ev.score_count(pred, split=split)
    print(f"[{split}] BINAER AUC={rb['auc']:.3f} AP={rb['ap']:.3f} | "
          f"COUNT MSE={rc['mse']:.3f} MAE={rc['mae']:.3f}")

## Nächste Schritte
- Werte gegen die Baselines stellen: **AUC/AP vs. GraphMixer**, **MSE/MAE vs. LSTM**.
- Optionale Hurdle-Verfeinerung: Count am Inferenzpunkt mit der Binär-Wahrscheinlichkeit
  gaten (`pred_count *= sigmoid(logit)`), falls die Nullen den MSE dominieren.
- Ablationen: nur Graph / nur Zeitreihe; GCN statt GraphSAGE; 1D-CNN statt GRU.